# HZ-CHORD-AI — stem separator training

This notebook matches the GitHub Actions pipeline. The model is waveform-in/waveform-out and exports directly to LiteRT/TFLite with the Android contract `[1, 352800] -> [1, 4, 352800]`.

Use a GPU runtime. A first run should use 1 epoch, batch size 1, and 1 segment per track to validate the pipeline before starting a long run.

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before training")
print("GPU:", torch.cuda.get_device_name(0))

## 1. Clone the repository

If the repository is private, authenticate Git before cloning. Do not put a personal access token directly into a notebook that will be shared.

In [ ]:
!git clone https://github.com/Zetthilly/HZ-CHORD-AI-pro.git /content/HZ-CHORD-AI-pro
%cd /content/HZ-CHORD-AI-pro

In [ ]:
!pip install -q -r training/stem_separator/requirements.txt

## 2. Download MUSDB18

The official MUSDB18 release is hosted on Zenodo. It is about 4.7 GB compressed.

In [ ]:
import os
MUSDB_DIR = "/content/musdb18"
if not os.path.exists(MUSDB_DIR + "/train"):
    !wget -q --show-progress -O /content/musdb18.zip https://zenodo.org/records/1117372/files/musdb18.zip
    !mkdir -p {MUSDB_DIR}
    !unzip -q /content/musdb18.zip -d {MUSDB_DIR}
print("Dataset:", MUSDB_DIR)

## 3. Smoke-test training

Run this first. It confirms MUSDB decoding, CUDA training and checkpoint creation without committing to a long run.

In [ ]:
!python training/stem_separator/lightweight_unet.py \
    --musdb-root {MUSDB_DIR} \
    --epochs 1 \
    --batch-size 1 \
    --segments-per-track 1 \
    --num-workers 2 \
    --checkpoint-path /content/lightweight_unet.pt

## 4. Real training

After the smoke test works, increase epochs and segments. Keep the checkpoint on Google Drive if using Colab so a disconnected session can resume.

In [ ]:
DRIVE_CKPT = "/content/lightweight_unet.pt"
# Example:
# !python training/stem_separator/lightweight_unet.py --musdb-root {MUSDB_DIR} --epochs 50 --batch-size 2 --segments-per-track 20 --checkpoint-path {DRIVE_CKPT} --resume-from {DRIVE_CKPT}

## 5. Export and validate TFLite

In [ ]:
!python training/stem_separator/export_to_tflite.py \
    --checkpoint /content/lightweight_unet.pt \
    --output /content/stem_separator.tflite \
    --chunk-samples 352800

In [ ]:
from google.colab import files
files.download("/content/stem_separator.tflite")